# MediDiary — Gemma 3 1B fine-tune

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
    !unsloth install-kernels
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.57.6
!pip install --no-deps trl==0.22.2

In [ ]:
BASE_MODEL      = "unsloth/gemma-3-1b-it"
MAX_SEQ_LEN     = 1024
SEED            = 3407

SYNTH_TRAIN_N   = 1000
SYNTH_VAL_N     = 80
ORIG_VAL_FRAC   = 0.15
ORIG_UPWEIGHT   = 2

LORA_R, LORA_ALPHA = 32, 32
EPOCHS          = 3
LR              = 1e-4
BATCH, GRAD_ACC = 4, 2

QUANT           = "q4_k_m"
GGUF_EVAL_N     = 30

RUN_NAME        = "medidiary_gemma3_1b_v2"
DRIVE_DIR       = f"/content/drive/MyDrive/{RUN_NAME}"
SAVE_TO_DRIVE   = True

import os
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)

In [ ]:
import json, random

SYSTEM_PROMPT = (
    'Extract medicine and symptom records from the transcript into strict JSON. '
    'Output only JSON with schema: {"records": [{"type": "medicine"|"symptom", '
    '"name": string, "generic_suggestion": null, '
    '"time_phrase": string|null, "note": string|null}]}. '
    'Keep unknown fields null. Never infer dose, strength, frequency, or missing time.'
)

def m(name, generic=None, time=None, note=None):
    return {"type": "medicine", "name": name, "generic_suggestion": None, "time_phrase": time, "note": note}

def s(name, time=None, note=None):
    return {"type": "symptom", "name": name, "generic_suggestion": None, "time_phrase": time, "note": note}

RAW_EXAMPLES = [
    ("সকালে একটা Napa খেয়েছি আর দুপুর থেকে একটু মাথাব্যথা করছে।", [m("Napa", "Paracetamol", "সকালে"), s("মাথাব্যথা", "দুপুর থেকে", "একটু")]),
    ("খালি পেটে একটা Seclo 20 খেয়েছি সকাল ৮টার দিকে।", [m("Seclo", "Omeprazole", "সকাল ৮টার দিকে", "খালি পেটে, 20")]),
    ("রাতে ঘুমানোর আগে Monas 10 খেয়েছিলাম, এখনো হালকা cough আছে।", [m("Monas", "Montelukast", "রাতে ঘুমানোর আগে", "10"), s("cough", "এখনো", "হালকা")]),
    ("দুপুরে lunch এর পর Ace খেয়েছি, জ্বরটা এখন কিছুটা কম।", [m("Ace", "Paracetamol", "দুপুরে lunch এর পর"), s("জ্বর", "এখন", "কিছুটা কম")]),
    ("সকাল থেকে খুব acidity feel হচ্ছে, তাই একটা Maxpro খেলাম।", [s("acidity", "সকাল থেকে", "খুব"), m("Maxpro", "Esomeprazole")]),
    ("গতকাল রাতে Fexo 120 খেয়েছি allergy-র জন্য, আজ অনেকটা ভালো।", [m("Fexo", "Fexofenadine", "গতকাল রাতে", "120"), s("allergy", "গতকাল রাতে")]),
    ("বিকেলে হঠাৎ মাথা ঘোরা শুরু হয়েছে, সাথে একটু weakness আছে।", [s("মাথা ঘোরা", "বিকেলে", "হঠাৎ"), s("weakness", "বিকেলে", "একটু")]),
    ("সকালে নাস্তার পর Comet 500 খেয়েছি আর প্রেসারের Bizoran টাও খেয়েছি।", [m("Comet", "Metformin", "সকালে নাস্তার পর", "500"), m("Bizoran", "Amlodipine + Olmesartan", "সকালে নাস্তার পর")]),
    ("দুই দিন ধরে dry cough হচ্ছে, রাতে ঠিকমতো ঘুম হচ্ছে না।", [s("dry cough", None, "দুই দিন ধরে"), s("ঘুম হচ্ছে না", "রাতে", "ঠিকমতো")]),
    ("সন্ধ্যায় একটা Napa Extra খেয়েছি severe headache এর জন্য।", [m("Napa Extra", "Paracetamol + Caffeine", "সন্ধ্যায়"), s("headache", "সন্ধ্যায়", "severe")]),
    ("ব্রেকফাস্টের আগে Sergel 20 খেয়েছি, বুক জ্বালাপোড়া এখনো হালকা আছে।", [m("Sergel", "Esomeprazole", "ব্রেকফাস্টের আগে", "20"), s("বুক জ্বালাপোড়া", "এখনো", "হালকা")]),
    ("আজকে ডিনার করার পর Pantonix 20 খেয়েছি।", [m("Pantonix", "Pantoprazole", "আজকে ডিনার করার পর", "20")]),
    ("সকাল থেকে নাক বন্ধ আর প্রচণ্ড সর্দি, রাতে Alatrol খাব ভাবছি।", [s("নাক বন্ধ", "সকাল থেকে"), s("সর্দি", "সকাল থেকে", "প্রচণ্ড")]),
    ("দুপুরে Azithrocin 500 খেয়েছি, গলা ব্যথা এখনো আছে।", [m("Azithrocin", "Azithromycin", "দুপুরে", "500"), s("গলা ব্যথা", "এখনো")]),
    ("পেটে খুব gas আর bloating হচ্ছে দুপুর থেকেই।", [s("gas", "দুপুর থেকেই", "খুব"), s("bloating", "দুপুর থেকেই", "খুব")]),
    ("সকালে একটা Orsaline খেয়েছি, শরীরটা অনেক দুর্বল লাগছে।", [m("Orsaline", "Oral Rehydration Salts", "সকালে"), s("শরীর দুর্বল", None, "অনেক")]),
    ("রাতে Tofen খেয়েছি, আজকে আর breathing problem হয়নি।", [m("Tofen", "Ketotifen", "রাতে")]),
    ("সকাল ১০টায় Amdocal 5 খেয়েছি, এখন ঘাড়ের দিকে একটু ব্যথা করছে।", [m("Amdocal", "Amlodipine", "সকাল ১০টায়", "5"), s("ঘাড়ে ব্যথা", "এখন", "একটু")]),
    ("খাবার আগে Losectil খেয়েছি আর খাবার পর Entacyd সিরাপ খেয়েছি।", [m("Losectil", "Omeprazole", "খাবার আগে"), m("Entacyd", None, "খাবার পর", "সিরাপ")]),
    ("আজ দুপুরে Ciprocin 500 নিয়েছি, একটু বমি বমি ভাব লাগছে।", [m("Ciprocin", "Ciprofloxacin", "আজ দুপুরে", "500"), s("বমি বমি ভাব", None, "একটু")]),
    ("সকালে Osartil 50 খেয়েছি, আজ কোনো dizziness নেই।", [m("Osartil", "Losartan", "সকালে", "50")]),
    ("গত রাত থেকে low-grade fever, সকালে শুধু একটা Ace Plus খেয়েছি।", [s("fever", "গত রাত থেকে", "low-grade"), m("Ace Plus", "Paracetamol + Caffeine", "সকালে")]),
    ("রাতে ডিনারের পর Atova 10 খেয়েছি।", [m("Atova", "Atorvastatin", "রাতে ডিনারের পর", "10")]),
    ("সকালে হাঁটার পর পায়ে একটু cramp হয়েছিল।", [s("পায়ে cramp", "সকালে হাঁটার পর", "একটু")]),
    ("দুপুরে খাবার পর Ecosprin 75 খেয়েছি।", [m("Ecosprin", "Aspirin", "দুপুরে খাবার পর", "75")]),
    ("আজকে Histacin খেয়েছি সকালের দিকে, এখন একটু ঘুম ঘুম ভাব হচ্ছে।", [m("Histacin", "Chlorpheniramine", "সকালের দিকে"), s("ঘুম ঘুম ভাব", "এখন", "একটু")]),
    ("দুপুর থেকে ডান চোখে একটু খচখচ করছে আর জল পড়ছে।", [s("চোখে খচখচ", "দুপুর থেকে", "ডান চোখে, একটু"), s("চোখ দিয়ে জল পড়া", "দুপুর থেকে")]),
    ("বিকেল ৫টায় একটা Napa 500 খেয়েছি body ache এর জন্য।", [m("Napa", "Paracetamol", "বিকেল ৫টায়", "500"), s("body ache", "বিকেল ৫টায়")]),
    ("গতকাল সন্ধ্যায় Maxpro 40 খেয়েছি, তবুও রাতে heartburn হয়েছিল।", [m("Maxpro", "Esomeprazole", "গতকাল সন্ধ্যায়", "40"), s("heartburn", "রাতে")]),
    ("সকালে একটা Neuro-B খেয়েছি, আজ হাত-পা ঝিনঝিন করা একটু কম।", [m("Neuro-B", None, "সকালে"), s("হাত-পা ঝিনঝিন", "আজ", "একটু কম")]),
    ("রাত ১১টার দিকে একটা Zimax খেয়েছি, কাশিটা এখনো পুরো যায়নি।", [m("Zimax", "Azithromycin", "রাত ১১টার দিকে"), s("কাশি", "এখনো", "পুরো যায়নি")]),
    ("সকাল থেকে mild chest tightness লাগছে, কোনো ওষুধ খাইনি এখনো।", [s("chest tightness", "সকাল থেকে", "mild")]),
    ("দুপুরে একটা Calbo-D খেয়েছি ভাতের পর।", [m("Calbo-D", None, "দুপুরে ভাতের পর")]),
    ("আধা ঘণ্টা আগে Adovas সিরাপ খেয়েছি, গলাটা এখনো খুসখুস করছে।", [m("Adovas", None, "আধা ঘণ্টা আগে", "সিরাপ"), s("গলা খুসখুস", "এখনো")]),
    ("আজ সকাল থেকে stomach pain হচ্ছে, তাই Sergel খেয়েছি।", [s("stomach pain", "আজ সকাল থেকে"), m("Sergel", "Esomeprazole")]),
    ("রাতের খাবারের পর Metformin 500 খেয়েছি।", [m("Metformin", "Metformin", "রাতের খাবারের পর", "500")]),
    ("সকালে নাস্তার সাথে একটা multivitamin খেয়েছি।", [m("multivitamin", None, "সকালে নাস্তার সাথে")]),
    ("গত দুই ঘণ্টা ধরে প্রচণ্ড migraine pain, তাই একটা Napa Extend খেলাম।", [s("migraine pain", "গত দুই ঘণ্টা ধরে", "প্রচণ্ড"), m("Napa Extend", "Paracetamol")]),
    ("সন্ধ্যা থেকে গা গরম লাগছে আর একটু শীত শীত করছে।", [s("গা গরম", "সন্ধ্যা থেকে"), s("শীত শীত করা", "সন্ধ্যা থেকে", "একটু")]),
    ("সকালে inhalers নিয়েছি দুই চাপ, শ্বাসকষ্ট আজকে অনেক কম।", [m("inhaler", None, "সকালে", "দুই চাপ"), s("শ্বাসকষ্ট", "আজকে", "অনেক কম")]),
    ("রাতে একটা নাপা খেয়েছি, এখনো হালকা জ্বর আছে।", [m("নাপা", "Paracetamol", "রাতে"), s("জ্বর", "এখনো", "হালকা")]),
    ("সকালে খালি পেটে একটা সেকলো খেয়েছি।", [m("সেকলো", "Omeprazole", "সকালে খালি পেটে")]),
    ("দুপুর থেকে আমার পেট ব্যথা করছে আর একবার পাতলা পায়খানা হয়েছে।", [s("পেট ব্যথা", "দুপুর থেকে"), s("পাতলা পায়খানা", "দুপুর থেকে", "একবার")]),
    ("গতকাল রাতে একটা অ্যাল্যাট্রল খেয়েছিলাম চুলকানির জন্য।", [m("অ্যাল্যাট্রল", "Cetirizine", "গতকাল রাতে"), s("চুলকানি", "গতকাল রাতে")]),
    ("আজ সকালে ভাত খাওয়ার পর একটা ম্যাক্সপ্রো খেয়েছি।", [m("ম্যাক্সপ্রো", "Esomeprazole", "আজ সকালে ভাত খাওয়ার পর")]),
    ("সকাল থেকে সারা গায়ে ব্যথা আর খুব ক্লান্ত লাগছে।", [s("গায়ে ব্যথা", "সকাল থেকে", "সারা গায়ে"), s("ক্লান্তি", "সকাল থেকে", "খুব")]),
    ("বিকেলে এক গ্লাস ওরস্যালাইন খেয়েছি।", [m("ওরস্যালাইন", "Oral Rehydration Salts", "বিকেলে", "এক গ্লাস")]),
    ("রাত ১০টায় মোনাস ট্যাবলেটটা খেয়েছি।", [m("মোনাস", "Montelukast", "রাত ১০টায়", "ট্যাবলেট")]),
    ("গত তিন দিন ধরে শুকনো কাশি, সাথে একটু শ্বাস নিতে কষ্ট হচ্ছে।", [s("শুকনো কাশি", None, "গত তিন দিন ধরে"), s("শ্বাস নিতে কষ্ট", None, "একটু")]),
    ("সকালে একটা ফ্যাক্সো খেয়েছি, হাঁচি এখন কিছুটা কমেছে।", [m("ফ্যাক্সো", "Fexofenadine", "সকালে"), s("হাঁচি", "এখন", "কিছুটা কমেছে")]),
    ("দুপুরে খাওয়ার আগে সার্জেল ২০ খেয়েছি।", [m("সার্জেল", "Esomeprazole", "দুপুরে খাওয়ার আগে", "২০")]),
    ("হঠাৎ করে মাথা ঝিমঝিম করছে আর চোখে অন্ধকার দেখছি।", [s("মাথা ঝিমঝিম", None, "হঠাৎ"), s("চোখে অন্ধকার দেখা", None)]),
    ("সকালে প্রেসারের ওষুধ খেয়েছি কিন্তু নামটা ঠিক মনে নেই।", [m("প্রেসারের ওষুধ", None, "সকালে")]),
    ("রাত থেকে কোমরে খুব ব্যথা হচ্ছে, হাঁটতে কষ্ট হচ্ছে।", [s("কোমরে ব্যথা", "রাত থেকে", "খুব"), s("হাঁটতে কষ্ট", "রাত থেকে")]),
    ("সকালে নাস্তা করে একটা কমেট খেয়েছি ডায়াবেটিসের জন্য।", [m("কমেট", "Metformin", "সকালে নাস্তা করে")]),
    ("সন্ধ্যায় একটা নাপা এক্সট্রা খেয়েছি মাথা ব্যথার জন্য।", [m("নাপা এক্সট্রা", "Paracetamol + Caffeine", "সন্ধ্যায়"), s("মাথা ব্যথা", "সন্ধ্যায়")]),
    ("দুপুর ২টার দিকে প্যানটোনিক্স খেয়েছি বুক জ্বলার জন্য।", [m("প্যানটোনিক্স", "Pantoprazole", "দুপুর ২টার দিকে"), s("বুক জ্বলা", "দুপুর ২টার দিকে")]),
    ("সকাল থেকে গলার স্বর ভেঙে গেছে আর ঢোক গিলতে ব্যথা লাগছে।", [s("গলার স্বর ভেঙে যাওয়া", "সকাল থেকে"), s("ঢোক গিলতে ব্যথা", "সকাল থেকে")]),
    ("রাতে কাশির সিরাপ খেয়েছি দুই চামচ।", [m("কাশির সিরাপ", None, "রাতে", "দুই চামচ")]),
    ("আজকে দুপুরের পর থেকে বমি ভাব হচ্ছে, খাবার খেতে ইচ্ছা করছে না।", [s("বমি ভাব", "আজকে দুপুরের পর থেকে"), s("ক্ষুধামন্দা", "আজকে দুপুরের পর থেকে")]),
    ("সকালে একটা অ্যান্টিবায়োটিক খেয়েছি, এজিত্রোসিন মনে হয় নাম।", [m("এজিত্রোসিন", "Azithromycin", "সকালে")]),
    ("বিকেল থেকে বাম কানে টনটন করে ব্যথা করছে।", [s("কানে ব্যথা", "বিকেল থেকে", "বাম কানে, টনটন")]),
    ("সকাল ৮টায় থাইরয়েডের ওষুধ খেয়েছি খালি পেটে।", [m("থাইরয়েডের ওষুধ", None, "সকাল ৮টায় খালি পেটে")]),
    ("আজকে রাতে একটা হিস্টাসিন খেয়েছি নাক দিয়ে পানি পড়ার জন্য।", [m("হিস্টাসিন", "Chlorpheniramine", "আজকে রাতে"), s("নাক দিয়ে পানি পড়া", "আজকে রাতে")]),
    ("দুপুর থেকে হাতের আঙুলের জয়েন্টে একটু ফোলা আর ব্যথা।", [s("আঙুলের জয়েন্টে ফোলা", "দুপুর থেকে", "একটু"), s("আঙুলের জয়েন্টে ব্যথা", "দুপুর থেকে", "একটু")]),
    ("রাতে ঘুমের ওষুধ অর্ধেকটা খেয়েছি।", [m("ঘুমের ওষুধ", None, "রাতে", "অর্ধেকটা")]),
    ("সকালে প্যারাসিটামল খেয়েছি, এখন শরীরের তাপমাত্রা স্বাভাবিক।", [m("প্যারাসিটামল", "Paracetamol", "সকালে")]),
    ("কয়েক ঘণ্টা ধরে বুকের বাম দিকে হালকা চাপ অনুভব করছি।", [s("বুকে চাপ অনুভব", None, "কয়েক ঘণ্টা ধরে, বাম দিকে, হালকা")]),
    ("সকালে চর্মরোগের মলমটা লাগিয়েছি, চুলকানি একটু কম আছে।", [m("চর্মরোগের মলম", None, "সকালে"), s("চুলকানি", None, "একটু কম")]),
    ("দুপুরে খাবারের পর এন্টাসিড ট্যাবলেট চিবিয়ে খেয়েছি।", [m("এন্টাসিড", None, "দুপুরে খাবারের পর", "ট্যাবলেট")]),
    ("Took Montelukast before sleep, mild headache since morning.", [m("Montelukast", "Montelukast", "before sleep"), s("headache", "since morning", "mild")]),
    ("Had one Napa 500mg after breakfast for body ache.", [m("Napa", "Paracetamol", "after breakfast", "500mg"), s("body ache", "after breakfast")]),
    ("Feeling dizzy and nauseous since 3 PM today.", [s("dizziness", "since 3 PM today"), s("nausea", "since 3 PM today")]),
    ("Took Seclo 20 on an empty stomach this morning.", [m("Seclo", "Omeprazole", "this morning", "empty stomach, 20")]),
    ("Sore throat and runny nose started last night.", [s("sore throat", "last night"), s("runny nose", "last night")]),
    ("Took Fexo 120 in the evening, sneezing stopped.", [m("Fexo", "Fexofenadine", "in the evening", "120")]),
    ("Mild fever and chills at midnight, took Ace.", [s("fever", "at midnight", "mild"), s("chills", "at midnight"), m("Ace", "Paracetamol", "at midnight")]),
    ("Took Metformin after dinner and Amlodipine in the morning.", [m("Metformin", "Metformin", "after dinner"), m("Amlodipine", "Amlodipine", "in the morning")]),
    ("Sharp pain in lower back since yesterday afternoon.", [s("lower back pain", "since yesterday afternoon", "sharp")]),
    ("Drank one packet of Orsaline around noon because of dehydration.", [m("Orsaline", "Oral Rehydration Salts", "around noon", "one packet"), s("dehydration", "around noon")]),
    ("Took Pantonix before lunch, still having slight heartburn.", [m("Pantonix", "Pantoprazole", "before lunch"), s("heartburn", None, "slight")]),
    ("No medicines taken, just feeling very tired and sleepy all day.", [s("tiredness", "all day", "very"), s("sleepiness", "all day")]),
    ("Took Alatrol at night for skin rash.", [m("Alatrol", "Cetirizine", "at night"), s("skin rash", "at night")]),
    ("Took Azithromycin 500mg after lunch.", [m("Azithromycin", "Azithromycin", "after lunch", "500mg")]),
    ("Eye irritation and redness in the right eye since waking up.", [s("eye irritation", "since waking up", "right eye"), s("eye redness", "since waking up", "right eye")]),
    ("আচ্ছা মানে সকালে নাপা এস্কট্রা খাইছি একটা মাথাটা খুব ধরছে এখনো", [m("নাপা এস্কট্রা", "Paracetamol + Caffeine", "সকালে"), s("মাথা ধরা", "এখনো", "খুব")]),
    ("হ্যালো সকালে সেরজেল ২০ খেলাম খালি পেটে", [m("সেরজেল", "Esomeprazole", "সকালে", "খালি পেটে, ২০")]),
    ("মানে কাল রাতে ওই কী যেন লাল রঙের ট্যাবলেটটা খেয়েছিলাম নাম মনে নাই গা ব্যথা ছিল", [m("লাল রঙের ট্যাবলেট", None, "কাল রাতে"), s("গা ব্যথা", "কাল রাতে")]),
    ("মনাস ১০ রাতের বেলা পেট একটু কামড়াচ্ছে এখন", [m("মনাস", "Montelukast", "রাতের বেলা", "১০"), s("পেট কামড়ানো", "এখন", "একটু")]),
    ("সকাল থেকে কাশি কাশি আর হাঁচি ফেক্সো ১২০ নিসি দুপুরে", [s("কাশি", "সকাল থেকে"), s("হাঁচি", "সকাল থেকে"), m("ফেক্সো", "Fexofenadine", "দুপুরে", "১২০")]),
    ("আজকে কোনো ওষুধ খাইনি", []),
    ("আজকে আলহামদুলিল্লাহ সব ঠিক আছে কোনো সমস্যা নাই", []),
    ("হ্যালো টেস্টিং ওয়ান টু থ্রি শোনা যাচ্ছে কি", []),
    ("কালকে ডাক্তারের কাছে যাব রিপোর্টগুলো দেখাতে হবে", []),
    ("ওষুধ শেষ হয়ে গেছে আজকে ফার্মেসি থেকে কিনতে হবে", []),
    ("Nothing to log today, feeling completely fine.", []),
    ("না আজকে আর কোনো জ্বর বা ব্যথা কিছু নেই", []),
    ("আব্বুকে সকালে একটা প্রেসারের ট্যাবলেট দিয়েছি নাম বিজরান ৫ বাই ২০", [m("বিজরান", "Amlodipine + Olmesartan", "সকালে", "৫ বাই ২০")]),
    ("পেটে গ্যাস লাগছিল তাই একটা সেকলো খেয়ে নিলাম কখন খেয়েছি মনে নেই", [s("পেটে গ্যাস", None), m("সেকলো", "Omeprazole")]),
    ("এখন খুব মাথা ঘুরছে বুক ধড়ফড় করছে", [s("মাথা ঘোরা", "এখন", "খুব"), s("বুক ধড়ফড়", "এখন")]),
    ("ফজরের পর Thyrox 50 খেয়েছি খালি পেটে।", [m("Thyrox", "Levothyroxine", "ফজরের পর", "50, খালি পেটে")]),
    ("সকাল থেকে খুব migraine উঠেছে, চোখেও আলো লাগছে।", [s("migraine", "সকাল থেকে", "খুব"), s("চোখে আলো লাগা", "সকাল থেকে")]),
    ("দুপুরে Omidon খেয়েছি খাওয়ার আগে, তবুও একটু nausea আছে।", [m("Omidon", "Domperidone", "দুপুরে খাওয়ার আগে"), s("nausea", None, "একটু")]),
    ("রাতে Bisolol 2.5 খেয়েছি, আজ heart rate স্বাভাবিক লাগছে।", [m("Bisolol", "Bisoprolol", "রাতে", "2.5")]),
    ("সকালে হাঁটুর joint এ খুব pain ছিল, তাই একটা Naprox 500 খেলাম।", [s("হাঁটুর joint এ pain", "সকালে", "খুব"), m("Naprox", "Naproxen", None, "500")]),
    ("গতকাল রাত থেকে dry throat আর হালকা body pain।", [s("dry throat", "গতকাল রাত থেকে"), s("body pain", "গতকাল রাত থেকে", "হালকা")]),
    ("খাওয়ার পর একটা Ceevit খেয়েছি আজ দুপুরে।", [m("Ceevit", "Ascorbic Acid", "আজ দুপুরে খাওয়ার পর")]),
    ("বিকেল থেকে lower abdomen এ mild cramp হচ্ছে।", [s("lower abdomen এ cramp", "বিকেল থেকে", "mild")]),
    ("সন্ধ্যায় একটা Rupal 10 খেয়েছি dust allergy-র জন্য।", [m("Rupal", "Rupatadine", "সন্ধ্যায়", "10"), s("dust allergy", "সন্ধ্যায়")]),
    ("রাত ৯টায় Clopid 75 নিয়েছি ডিনারের পর।", [m("Clopid", "Clopidogrel", "রাত ৯টায় ডিনারের পর", "75")]),
    ("সকালে আম্মুকে Secrin 2 দিয়েছি নাস্তার আগে।", [m("Secrin", "Glimepiride", "সকালে নাস্তার আগে", "2")]),
    ("দুপুর থেকে বুকে একটু heaviness লাগছে আর শ্বাস নিতে কষ্ট হচ্ছে।", [s("বুকে heaviness", "দুপুর থেকে", "একটু"), s("শ্বাস নিতে কষ্ট", "দুপুর থেকে")]),
    ("আজ রাতে D-Rise ক্যাপসুল খেয়েছি ভাতের পর।", [m("D-Rise", "Cholecalciferol", "আজ রাতে ভাতের পর", "ক্যাপসুল")]),
    ("সকাল ১১টায় একটা Emistat খেয়েছি, বমিটা এখন বন্ধ হয়েছে।", [m("Emistat", "Ondansetron", "সকাল ১১টায়"), s("বমি", "সকাল ১১টায়", "এখন বন্ধ")]),
    ("গত দুই দিন ধরে neck stiffness আর ঘাড় ঘোরানো যাচ্ছে না।", [s("neck stiffness", None, "গত দুই দিন ধরে"), s("ঘাড় ঘোরানো যাচ্ছে না", None, "গত দুই দিন ধরে")]),
    ("রাতে ঘুমের আগে Tamsin ক্যাপসুল খেয়েছি।", [m("Tamsin", "Tamsulosin", "রাতে ঘুমের আগে", "ক্যাপসুল")]),
    ("সকালে একটা Cef-3 400 খেয়েছি throat infection এর জন্য।", [m("Cef-3", "Cefixime", "সকালে", "400"), s("throat infection", "সকালে")]),
    ("দুপুরে প্রচণ্ড acidity হয়েছিল, Gavisol সিরাপ দুই চামচ খেয়েছি।", [s("acidity", "দুপুরে", "প্রচণ্ড"), m("Gavisol", None, "দুপুরে", "সিরাপ, দুই চামচ")]),
    ("আজ সন্ধ্যায় হঠাৎ nosebleed হয়েছিল পাঁচ মিনিট মতো।", [s("nosebleed", "আজ সন্ধ্যায়", "হঠাৎ, পাঁচ মিনিট মতো")]),
    ("সকালে রোদে যাওয়ার পর থেকেই skin itching শুরু হয়েছে।", [s("skin itching", "সকালে রোদে যাওয়ার পর থেকে")]),
    ("রাতে খাবার পর একটা Rosuva 10 খেয়েছি।", [m("Rosuva", "Rosuvastatin", "রাতে খাবার পর", "10")]),
    ("সকাল থেকে পায়ের পাতায় একটু swelling দেখছি।", [s("পায়ের পাতায় swelling", "সকাল থেকে", "একটু")]),
    ("দুপুর ১টার দিকে একটা Filmet 400 খেয়েছি stomach upset এর জন্য।", [m("Filmet", "Metronidazole", "দুপুর ১টার দিকে", "400"), s("stomach upset", "দুপুর ১টার দিকে")]),
    ("সন্ধ্যা থেকে continuous sneezing হচ্ছে, চোখেও জল আসছে।", [s("sneezing", "সন্ধ্যা থেকে", "continuous"), s("চোখে জল আসা", "সন্ধ্যা থেকে")]),
    ("আজকে সকালে Bextram Gold খেয়েছি নাস্তার পর।", [m("Bextram Gold", None, "আজকে সকালে নাস্তার পর")]),
    ("রাত ১২টার পর থেকে দাঁতে severe pain শুরু হয়েছে।", [s("দাঁতে pain", "রাত ১২টার পর থেকে", "severe")]),
    ("সকালে একটা Algin খেয়েছি পেট কামড়ানোর জন্য।", [m("Algin", None, "সকালে"), s("পেট কামড়ানো", "সকালে")]),
    ("দুপুরে খাবারের পর থেকে খুব drowsiness লাগছে।", [s("drowsiness", "দুপুরে খাবারের পর থেকে", "খুব")]),
    ("গতকাল সন্ধ্যায় Deslor 5 খেয়েছি, আজ আর ঠান্ডা লাগছে না।", [m("Deslor", "Desloratadine", "গতকাল সন্ধ্যায়", "5")]),
    ("সকালে ১ চামচ Tuska সিরাপ খেয়েছি শুকনো কাশির জন্য।", [m("Tuska", None, "সকালে", "সিরাপ, ১ চামচ"), s("শুকনো কাশি", "সকালে")]),
    ("বিকেল থেকে বাম পায়ের calf muscle এ pain হচ্ছে।", [s("calf muscle এ pain", "বিকেল থেকে", "বাম পায়ের")]),
    ("রাতে একটা Ace 500 নিয়েছি হালকা fever feel হচ্ছিল।", [m("Ace", "Paracetamol", "রাতে", "500"), s("fever", "রাতে", "হালকা")]),
    ("সকালে নাস্তার আগে Nexum 20 খেয়েছি।", [m("Nexum", "Esomeprazole", "সকালে নাস্তার আগে", "20")]),
    ("দুই ঘণ্টা ধরে একটু blurred vision মনে হচ্ছে ডান চোখে।", [s("blurred vision", "দুই ঘণ্টা ধরে", "ডান চোখে, একটু")]),
    ("রাতে খাবার পর একটা Zinc ট্যাবলেট খেয়েছি।", [m("Zinc", "Zinc", "রাতে খাবার পর", "ট্যাবলেট")]),
    ("সকালে বাবুকে Napa syrup খাইয়েছি এক চামচ, ওর গা গরম ছিল।", [m("Napa", "Paracetamol", "সকালে", "syrup, এক চামচ"), s("গা গরম", "সকালে")]),
    ("গতকাল দুপুর থেকে কান ভোঁ ভোঁ করছে আর একটু hearing problem লাগছে।", [s("কান ভোঁ ভোঁ করা", "গতকাল দুপুর থেকে"), s("hearing problem", "গতকাল দুপুর থেকে", "একটু")]),
    ("আজ সকালে Amaryl 1 খেয়েছি ব্রেকফাস্টের ঠিক আগে।", [m("Amaryl", "Glimepiride", "আজ সকালে ব্রেকফাস্টের ঠিক আগে", "1")]),
    ("সন্ধ্যায় একটা Rolac 10 খেয়েছি কোমরের ব্যথার জন্য।", [m("Rolac", "Ketorolac", "সন্ধ্যায়", "10"), s("কোমরের ব্যথা", "সন্ধ্যায়")]),
    ("রাত থেকে mouth ulcer টা খুব জ্বলছে, খেতে পারছি না।", [s("mouth ulcer", "রাত থেকে", "খুব জ্বলছে")]),
    ("সকালে Camlodin 5 খেয়েছি, প্রেসার এখন নরমাল।", [m("Camlodin", "Amlodipine", "সকালে", "5")]),
    ("দুপুরে একটু loose motion হয়েছে দুইবার, সাথে Orsaline খেয়েছি।", [s("loose motion", "দুপুরে", "দুইবার, একটু"), m("Orsaline", "Oral Rehydration Salts", "দুপুরে")]),
    ("আজ রাতে Fenadin 120 খেয়েছি হাতের rash এর জন্য।", [m("Fenadin", "Fexofenadine", "আজ রাতে", "120"), s("হাতের rash", "আজ রাতে")]),
    ("সকাল থেকে কাঁধে stiffness আর হাত তুলতে গেলে লাগছে।", [s("কাঁধে stiffness", "সকাল থেকে"), s("হাত তুলতে ব্যথা", "সকাল থেকে")]),
    ("বিকেল ৪টায় একটা Napa Extra খেয়েছি, অফিসের কাজ করতে করতে মাথা ধরেছিল।", [m("Napa Extra", "Paracetamol + Caffeine", "বিকেল ৪টায়"), s("মাথা ধরা", "বিকেল ৪টায়")]),
    ("রাতে ভাতের পর Lipicon 20 খেয়েছি।", [m("Lipicon", "Atorvastatin", "রাতে ভাতের পর", "20")]),
    ("সকাল থেকে বুকটা একটু জ্বালা করছে আর টক ঢেঁকুর উঠছে।", [s("বুক জ্বালা", "সকাল থেকে", "একটু"), s("টক ঢেঁকুর", "সকাল থেকে")]),
    ("গতকাল রাতে Motigut খেয়েছিলাম, পেট ফাঁপা এখনো পুরো কমেনি।", [m("Motigut", "Domperidone", "গতকাল রাতে"), s("পেট ফাঁপা", "এখনো", "পুরো কমেনি")]),
    ("সকালে একটা Fast 500 খেয়েছি গা ম্যাজম্যাজ করার জন্য।", [m("Fast", "Paracetamol", "সকালে", "500"), s("গা ম্যাজম্যাজ", "সকালে")]),
    ("দুপুর থেকে হালকা earache আর একটু জ্বর জ্বর ভাব।", [s("earache", "দুপুর থেকে", "হালকা"), s("জ্বর জ্বর ভাব", "দুপুর থেকে", "একটু")]),
    ("রাতের খাবারের পর Montair 10 খেয়েছি।", [m("Montair", "Montelukast", "রাতের খাবারের পর", "10")]),
    ("সকালে নাস্তার পর থেকে হাত কাঁপছে আর একটু sweating হচ্ছে।", [s("হাত কাঁপা", "সকালে নাস্তার পর থেকে"), s("sweating", "সকালে নাস্তার পর থেকে", "একটু")]),
    ("দুপুরে একটা Fimaxil 500 খেয়েছি দাঁতের ইনফেকশনের জন্য।", [m("Fimaxil", "Amoxicillin", "দুপুরে", "500"), s("দাঁতের ইনফেকশন", "দুপুরে")]),
    ("আজ সন্ধ্যায় একটু palpitation হয়েছিল সিঁড়ি ভাঙার পর।", [s("palpitation", "আজ সন্ধ্যায় সিঁড়ি ভাঙার পর", "একটু")]),
    ("সকালে খালি পেটে Pantodac 40 খেয়েছি।", [m("Pantodac", "Pantoprazole", "সকালে খালি পেটে", "40")]),
    ("দুই দিন ধরে heel pain হচ্ছে সকালে ঘুম থেকে ওঠার সময়।", [s("heel pain", "সকালে ঘুম থেকে ওঠার সময়", "দুই দিন ধরে")]),
    ("রাতে একটা Viseral খেয়েছি, পেটের মোচড়টা এখন অনেক কম।", [m("Viseral", None, "রাতে"), s("পেটের মোচড়", "এখন", "অনেক কম")]),
    ("সকাল থেকে প্রচণ্ড nasal congestion, কথা বলতে কষ্ট হচ্ছে।", [s("nasal congestion", "সকাল থেকে", "প্রচণ্ড")]),
    ("দুপুরে একটা Amodis 400 খেয়েছি আমাশয়ের জন্য।", [m("Amodis", "Metronidazole", "দুপুরে", "400"), s("আমাশয়", "দুপুরে")]),
    ("রাতে ঘুমানোর সময় পায়ের পেশিতে হঠাৎ টান লেগেছিল।", [s("পায়ের পেশিতে টান", "রাতে ঘুমানোর সময়", "হঠাৎ")]),
    ("সকালে Glucophage 500 খেয়েছি নাস্তার সাথে।", [m("Glucophage", "Metformin", "সকালে নাস্তার সাথে", "500")]),
    ("আজকে দুপুর থেকে গলায় কিছু আটকে থাকার মতো feeling হচ্ছে।", [s("গলায় কিছু আটকে থাকার অনুভূতি", "আজকে দুপুর থেকে")]),
    ("সন্ধ্যায় একটা Seclo 40 খেয়েছি, গ্যাস্ট্রিকটা খুব বাড়ছিল।", [m("Seclo", "Omeprazole", "সন্ধ্যায়", "40"), s("গ্যাস্ট্রিক", "সন্ধ্যায়", "খুব")]),
    ("সকালে চোখে artificial tears দিয়েছি দুই ফোঁটা, চোখ খুব dry ছিল।", [m("artificial tears", None, "সকালে", "দুই ফোঁটা"), s("dry eye", "সকালে", "খুব")]),
    ("রাত ৮টায় একটা Paracetamol খেয়েছি দাঁতের ব্যথা কমানোর জন্য।", [m("Paracetamol", "Paracetamol", "রাত ৮টায়"), s("দাঁতের ব্যথা", "রাত ৮টায়")]),
    ("গতকাল বিকেল থেকে হাতের কনুইয়ে একটু swelling আর লালচে ভাব।", [s("কনুইয়ে swelling", "গতকাল বিকেল থেকে", "হাতের কনুইয়ে, একটু"), s("কনুইয়ে লালচে ভাব", "গতকাল বিকেল থেকে")]),
    ("সকালে খাবার পর একটা Alatrol খেয়েছি, চুলকানি এখন নেই বললেই চলে।", [m("Alatrol", "Cetirizine", "সকালে খাবার পর"), s("চুলকানি", "এখন", "নেই বললেই চলে")]),
    ("দুপুরে খুব fatigue লাগছিল, কোনো কাজেই মন বসছিল না।", [s("fatigue", "দুপুরে", "খুব")]),
    ("রাতে খাবার আগে একটা Losectil 20 খেয়েছি।", [m("Losectil", "Omeprazole", "রাতে খাবার আগে", "20")]),
    ("সকাল থেকে হালকা runny nose আর মাথার সামনে ভার ভার লাগছে।", [s("runny nose", "সকাল থেকে", "হালকা"), s("মাথা ভার লাগা", "সকাল থেকে", "মাথার সামনে")]),
    ("বিকেল ৩টার দিকে একটা Ace খেয়েছি, জ্বরটা ১০১ এর মতো ছিল।", [m("Ace", "Paracetamol", "বিকেল ৩টার দিকে"), s("জ্বর", "বিকেল ৩টার দিকে", "১০১ এর মতো")]),
    ("গত রাত থেকে হাঁচির সাথে একটু chest wheezing হচ্ছে।", [s("হাঁচি", "গত রাত থেকে"), s("chest wheezing", "গত রাত থেকে", "একটু")]),
    ("সকালে নাস্তার পর একটা calcium tablet খেয়েছি।", [m("calcium tablet", None, "সকালে নাস্তার পর")]),
    ("দুপুরে ঝাল খাওয়ার পর থেকেই প্রচণ্ড heartburn শুরু হয়েছে।", [s("heartburn", "দুপুরে ঝাল খাওয়ার পর থেকে", "প্রচণ্ড")]),
    ("রাতে ঘুমের মধ্যে দুইবার কাশির জন্য ঘুম ভেঙে গেছে।", [s("কাশি", "রাতে ঘুমের মধ্যে", "দুইবার")]),
    ("সকালে একটা Maxpro 20 খেয়েছি, আজকে পেট অনেকটা আরাম।", [m("Maxpro", "Esomeprazole", "সকালে", "20")]),
    ("বিকেল থেকে পিঠের ডান পাশে হালকা চিনচিনে ব্যথা।", [s("পিঠে ব্যথা", "বিকেল থেকে", "ডান পাশে, হালকা, চিনচিনে")]),
    ("রাত ১০টায় Fexo 180 খেয়েছি, সারা গায়ে চাকা চাকা হয়ে চুলকাচ্ছিল।", [m("Fexo", "Fexofenadine", "রাত ১০টায়", "180"), s("চুলকানি ও চাকা", "রাত ১০টায়", "সারা গায়ে")]),
    ("সকালে খালি পেটে গ্যাস্ট্রিকের ওষুধটা খেয়েছি, নাম সার্জেল।", [m("সার্জেল", "Esomeprazole", "সকালে খালি পেটে")]),
    ("দুপুর থেকে শরীরটা কেমন যেন ঝিমুচ্ছে আর হালকা মাথা ব্যথা।", [s("শরীর ঝিমঝিম করা", "দুপুর থেকে"), s("মাথা ব্যথা", "দুপুর থেকে", "হালকা")]),
    ("সকালে নাস্তার আগে একটা প্যানটোনিক্স খেয়েছি।", [m("প্যানটোনিক্স", "Pantoprazole", "সকালে নাস্তার আগে")]),
    ("দুপুর থেকে আমার বাম হাতের কব্জিতে খুব ব্যথা করছে।", [s("হাতের কব্জিতে ব্যথা", "দুপুর থেকে", "বাম হাত, খুব")]),
    ("রাতে ভাত খাওয়ার পর আব্বুকে বিজরান দিয়েছি।", [m("বিজরান", "Amlodipine + Olmesartan", "রাতে ভাতের পর")]),
    ("আজকে সকাল থেকে মুখে কোনো স্বাদ পাচ্ছি না আর গন্ধও পাচ্ছি না।", [s("মুখে স্বাদ না পাওয়া", "আজকে সকাল থেকে"), s("গন্ধ না পাওয়া", "আজকে সকাল থেকে")]),
    ("বিকেলে একটা নাপা খেয়েছি, গায়ে একটু ব্যথা ছিল।", [m("নাপা", "Paracetamol", "বিকেলে"), s("গায়ে ব্যথা", "বিকেলে", "একটু")]),
    ("রাত থেকে পেটের ডান দিকে মৃদু ব্যথা হচ্ছে।", [s("পেটে ব্যথা", "রাত থেকে", "ডান দিকে, মৃদু")]),
    ("সকালে খালি পেটে এক গ্লাস ইসবগুলের ভুসি খেয়েছি।", [m("ইসবগুলের ভুসি", None, "সকালে খালি পেটে", "এক গ্লাস")]),
    ("দুপুরে একটা এইস ট্যাবলেট খেয়েছি জ্বরের জন্য।", [m("এইস", "Paracetamol", "দুপুরে", "ট্যাবলেট"), s("জ্বর", "দুপুরে")]),
    ("সন্ধ্যা থেকে ঘন ঘন প্রস্রাব হচ্ছে আর একটু জ্বালাপোড়া আছে।", [s("ঘন ঘন প্রস্রাব", "সন্ধ্যা থেকে"), s("প্রস্রাবে জ্বালাপোড়া", "সন্ধ্যা থেকে", "একটু")]),
    ("রাতে ঘুমানোর আগে একটা টোফেন খেয়েছি।", [m("টোফেন", "Ketotifen", "রাতে ঘুমানোর আগে")]),
    ("সকালে হাঁটার সময় হঠাৎ বুকে একটু চিনচিন ব্যথা অনুভব করেছি।", [s("বুকে ব্যথা", "সকালে হাঁটার সময়", "হঠাৎ, একটু, চিনচিন")]),
    ("আজকে দুপুরে খাবার পর এটোভা ১০ খেয়েছি।", [m("এটোভা", "Atorvastatin", "আজকে দুপুরে খাবার পর", "১০")]),
    ("গতকাল থেকে ঠোঁটের কোণে একটা ঘা হয়েছে, খুব ব্যথা করে।", [s("ঠোঁটের কোণে ঘা", None, "গতকাল থেকে, খুব ব্যথা")]),
    ("সকাল ৯টার দিকে অ্যামডোকাল ৫ খেয়েছি।", [m("অ্যামডোকাল", "Amlodipine", "সকাল ৯টার দিকে", "৫")]),
    ("বিকেল থেকে মাথায় প্রচণ্ড যন্ত্রণা হচ্ছে, বমিও হয়েছে একবার।", [s("মাথায় যন্ত্রণা", "বিকেল থেকে", "প্রচণ্ড"), s("বমি", "বিকেল থেকে", "একবার")]),
    ("রাতে খাওয়ার আধা ঘণ্টা আগে লোসেক্টিল ২০ খেয়েছি।", [m("লোসেক্টিল", "Omeprazole", "রাতে খাওয়ার আধা ঘণ্টা আগে", "২০")]),
    ("সকালে এক চামচ বাসক সিরাপ খেয়েছি গলা ব্যথার জন্য।", [m("বাসক", None, "সকালে", "সিরাপ, এক চামচ"), s("গলা ব্যথা", "সকালে")]),
    ("দুপুর থেকে ডান পায়ের গোড়ালিতে বেশ ব্যথা লাগছে।", [s("পায়ের গোড়ালিতে ব্যথা", "দুপুর থেকে", "ডান পা, বেশ")]),
    ("আজকে সকালে সিপ্রোসিন ৫০০ খেয়েছি নাস্তার পর।", [m("সিপ্রোসিন", "Ciprofloxacin", "আজকে সকালে নাস্তার পর", "৫০০")]),
    ("রাত ২টার দিকে হঠাৎ শ্বাসকষ্টে ঘুম ভেঙে গিয়েছিল।", [s("শ্বাসকষ্ট", "রাত ২টার দিকে", "হঠাৎ")]),
    ("সকালে একটা ইকোস্প্রিন খেয়েছি নাস্তার পর।", [m("ইকোস্প্রিন", "Aspirin", "সকালে নাস্তার পর")]),
    ("দুপুর থেকে চোখ দুটি খুব লাল হয়ে আছে আর চুলকাচ্ছে।", [s("চোখ লাল হওয়া", "দুপুর থেকে", "দুই চোখ, খুব"), s("চোখ চুলকানো", "দুপুর থেকে")]),
    ("রাতে ভাতের সাথে একটা ভিটামিন সি ট্যাবলেট খেয়েছি।", [m("ভিটামিন সি", "Ascorbic Acid", "রাতে ভাতের সাথে", "ট্যাবলেট")]),
    ("সকাল থেকে কোষ্ঠকাঠিন্য মনে হচ্ছে, পেট পরিষ্কার হয়নি।", [s("কোষ্ঠকাঠিন্য", "সকাল থেকে")]),
    ("দুপুর ১২টায় একটা ওমিডন খেয়েছি খাওয়ার আগে।", [m("ওমিডন", "Domperidone", "দুপুর ১২টায় খাওয়ার আগে")]),
    ("সন্ধ্যা থেকে শরীরটা ভারী লাগছে আর গায়ে একটু তাপ আছে।", [s("শরীর ভারী লাগা", "সন্ধ্যা থেকে"), s("জ্বর", "সন্ধ্যা থেকে", "একটু")]),
    ("সকালে ডায়াবেটিসের ইনসুলিন নিয়েছি দশ ইউনিট নাস্তার আগে।", [m("ইনসুলিন", None, "সকালে নাস্তার আগে", "দশ ইউনিট")]),
    ("গতকাল রাত থেকে পিঠের মাঝখানে টান লেগে আছে।", [s("পিঠে টান লাগা", "গতকাল রাত থেকে", "মাঝখানে")]),
    ("দুপুরে একটা ফ্লাজিল খেয়েছি পেট খারাপের জন্য।", [m("ফ্লাজিল", "Metronidazole", "দুপুরে"), s("পেট খারাপ", "দুপুরে")]),
    ("বিকেল থেকে খুব পিপাসা পাচ্ছে আর মুখ শুকিয়ে আসছে।", [s("পিপাসা", "বিকেল থেকে", "খুব"), s("মুখ শুকিয়ে আসা", "বিকেল থেকে")]),
    ("রাতে ঘুমানোর সময় একটা অ্যালজিন খেয়েছি পেট ব্যথায়।", [m("অ্যালজিন", None, "রাতে ঘুমানোর সময়"), s("পেট ব্যথা", "রাতে ঘুমানোর সময়")]),
    ("সকাল থেকে পায়ের পাতা দুটো কেমন যেন অবশ অবশ লাগছে।", [s("পায়ের পাতা অবশ লাগা", "সকাল থেকে")]),
    ("আজকে সকালে ওসার্টিল ৫০ খেয়েছি।", [m("ওসার্টিল", "Losartan", "আজকে সকালে", "৫০")]),
    ("দুপুরের পর থেকে কপালে দপদপ করে ব্যথা করছে।", [s("কপালে ব্যথা", "দুপুরের পর থেকে", "দপদপ")]),
    ("রাতে একটা রসুভা ৫ খেয়েছি।", [m("রসুভা", "Rosuvastatin", "রাতে", "৫")]),
    ("সকাল থেকে একটানা হেঁচকি উঠছে আর পেটে অস্বস্তি লাগছে।", [s("হেঁচকি", "সকাল থেকে", "একটানা"), s("পেটে অস্বস্তি", "সকাল থেকে")]),
    ("বিকেলে এক প্যাকেট ওরস্যালাইন পানিতে গুলে খেয়েছি।", [m("ওরস্যালাইন", "Oral Rehydration Salts", "বিকেলে", "এক প্যাকেট")]),
    ("গতকাল থেকে গলার নিচে একটু ফোলা ফোলা লাগছে।", [s("গলার নিচে ফোলা", None, "গতকাল থেকে, একটু")]),
    ("সকালে একটা ক্লোপিড খেয়েছি খাবারের পর।", [m("ক্লোপিড", "Clopidogrel", "সকালে খাবারের পর")]),
    ("রাত ১১টায় একটা নাপা খেয়েছি, এখনো ঘাম দিয়ে জ্বর ছাড়েনি।", [m("নাপা", "Paracetamol", "রাত ১১টায়"), s("জ্বর", "এখনো", "ছাড়েনি")]),
    ("সকালে ঘুম থেকে উঠে দেখি ঘাড়ের পেশি শক্ত হয়ে আছে।", [s("ঘাড়ের পেশি শক্ত হয়ে যাওয়া", "সকালে ঘুম থেকে উঠে")]),
    ("দুপুরে একটা সেফ-৩ ক্যাপসুল খেয়েছি।", [m("সেফ-৩", "Cefixime", "দুপুরে", "ক্যাপসুল")]),
    ("সন্ধ্যায় বাম হাঁটুতে একটু বরফ দিয়েছি, ফোলা এখনো আছে।", [s("হাঁটু ফোলা", "এখনো", "বাম হাঁটু")]),
    ("সকালে নাস্তার পর থাইরক্স ২৫ খেয়েছি।", [m("থাইরক্স", "Levothyroxine", "সকালে নাস্তার পর", "২৫")]),
    ("রাত থেকে নাকে রক্তের মতো একটু দাগ দেখলাম ঝাড়ার সময়।", [s("নাকে রক্তের দাগ", "রাত থেকে", "একটু")]),
    ("আজকে দুপুরে মটিগাট খেয়েছি বমি ভাব কমানোর জন্য।", [m("মটিগাট", "Domperidone", "আজকে দুপুরে"), s("বমি ভাব", "আজকে দুপুরে")]),
    ("সকাল থেকে সারা শরীরে ছোট ছোট লাল দানা বের হয়েছে।", [s("শরীরে লাল দানা", "সকাল থেকে", "সারা শরীরে, ছোট ছোট")]),
    ("বিকেলে ভুলে দুইটা সেকলো খেয়ে ফেলেছি।", [m("সেকলো", "Omeprazole", "বিকেলে", "দুইটা")]),
    ("গত দুই রাত ধরে পায়ের তলায় খুব জ্বালাপোড়া হচ্ছে।", [s("পায়ের তলায় জ্বালাপোড়া", None, "গত দুই রাত ধরে, খুব")]),
    ("সকালে একটা ফাস্ট ট্যাবলেট খেয়েছি মাথা ধরার জন্য।", [m("ফাস্ট", "Paracetamol", "সকালে", "ট্যাবলেট"), s("মাথা ধরা", "সকালে")]),
    ("দুপুরে খাবারের পর পেটে খুব গ্যাস জমেছিল।", [s("পেটে গ্যাস", "দুপুরে খাবারের পর", "খুব")]),
    ("রাতে আদোভাস সিরাপ খেয়েছি দুই চামচ।", [m("আদোভাস", None, "রাতে", "সিরাপ, দুই চামচ")]),
    ("সকাল থেকে মেজাজ খুব খিটখিটে লাগছে আর চোখে ঘুম নেই।", [s("ঘুম না আসা", "সকাল থেকে")]),
    ("দুপুর ৩টার দিকে একটা জিম্যাক্স ৫০০ খেয়েছি।", [m("জিম্যাক্স", "Azithromycin", "দুপুর ৩টার দিকে", "৫০০")]),
    ("সন্ধ্যায় হাঁটার পর একটু হাঁপানি শুরু হয়েছিল।", [s("হাঁপানি", "সন্ধ্যায় হাঁটার পর", "একটু")]),
    ("সকালে একটা এইস প্লাস খেয়েছি মাইগ্রেনের ব্যথায়।", [m("এইস প্লাস", "Paracetamol + Caffeine", "সকালে"), s("মাইগ্রেনের ব্যথা", "সকালে")]),
    ("রাত থেকে প্রস্রাবের পরিমাণ খুব কম মনে হচ্ছে।", [s("প্রস্রাবের পরিমাণ কম", "রাত থেকে", "খুব")]),
    ("দুপুরে খাবার পর একটা জিঙ্ক ট্যাবলেট খেয়েছি।", [m("জিঙ্ক", "Zinc", "দুপুরে খাবার পর", "ট্যাবলেট")]),
    ("সকাল থেকে বাম কানের ভেতর ভোঁ ভোঁ শব্দ হচ্ছে।", [s("কানের ভেতর শব্দ", "সকাল থেকে", "বাম কান, ভোঁ ভোঁ")]),
    ("রাতে ঘুমানোর আগে একটা রুপা ১০ খেয়েছি।", [m("রুপা", "Rupatadine", "রাতে ঘুমানোর আগে", "১০")]),
    ("Took Bizoran after breakfast and experienced mild dizziness around 11 AM.", [m("Bizoran", "Amlodipine + Olmesartan", "after breakfast"), s("dizziness", "around 11 AM", "mild")]),
    ("Had Sergel 20 before dinner, no acid reflux tonight.", [m("Sergel", "Esomeprazole", "before dinner", "20")]),
    ("Woke up with a stiff neck and slight fever this morning.", [s("stiff neck", "this morning"), s("fever", "this morning", "slight")]),
    ("Took Napa Extra in the afternoon for a throbbing headache.", [m("Napa Extra", "Paracetamol + Caffeine", "in the afternoon"), s("headache", "in the afternoon", "throbbing")]),
    ("Experiencing dry cough and shortness of breath since evening.", [s("dry cough", "since evening"), s("shortness of breath", "since evening")]),
    ("Gave my mother Losartan 50mg at 8 AM.", [m("Losartan", "Losartan", "at 8 AM", "50mg")]),
    ("Took Ciprocin 500 after lunch; having slight stomach cramps now.", [m("Ciprocin", "Ciprofloxacin", "after lunch", "500"), s("stomach cramps", "now", "slight")]),
    ("Applied anti-fungal cream on the ankle rash before bed.", [m("anti-fungal cream", None, "before bed"), s("ankle rash", "before bed")]),
    ("Persistent bloating and burping since yesterday night.", [s("bloating", "since yesterday night", "persistent"), s("burping", "since yesterday night", "persistent")]),
    ("Took Atova 10 after dinner as usual.", [m("Atova", "Atorvastatin", "after dinner", "10")]),
    ("Feeling nauseous after taking the morning multivitamin.", [s("nausea", "after taking the morning multivitamin"), m("multivitamin", None, "morning")]),
    ("Had one Omidon tablet before lunch for stomach discomfort.", [m("Omidon", "Domperidone", "before lunch", "tablet"), s("stomach discomfort", "before lunch")]),
    ("Sharp pain in the left knee when climbing stairs this evening.", [s("knee pain", "this evening", "left knee, sharp")]),
    ("Took Histacin at noon, felt very drowsy afterward.", [m("Histacin", "Chlorpheniramine", "at noon"), s("drowsiness", "afterward", "very")]),
    ("Mild sore throat and fatigue since waking up today.", [s("sore throat", "since waking up today", "mild"), s("fatigue", "since waking up today")]),
    ("Took two puffs of the inhaler around 4 PM for wheezing.", [m("inhaler", None, "around 4 PM", "two puffs"), s("wheezing", "around 4 PM")]),
    ("Took Comet 500 after breakfast and Comet 500 after dinner.", [m("Comet", "Metformin", "after breakfast", "500"), m("Comet", "Metformin", "after dinner", "500")]),
    ("Slight ear pain and blocked sensation on the right side since noon.", [s("ear pain", "since noon", "right side, slight"), s("blocked ear", "since noon", "right side")]),
    ("Had Entacyd chewable tablet after spicy dinner.", [m("Entacyd", None, "after spicy dinner", "chewable tablet")]),
    ("No fever today, just mild weakness in the afternoon.", [s("weakness", "in the afternoon", "mild")]),
    ("Took Clopid 75 and Ecosprin 75 after lunch.", [m("Clopid", "Clopidogrel", "after lunch", "75"), m("Ecosprin", "Aspirin", "after lunch", "75")]),
    ("Sudden muscle cramp in the right calf around 3 AM.", [s("muscle cramp", "around 3 AM", "sudden, right calf")]),
    ("Took Rupal 10 before sleep for hives.", [m("Rupal", "Rupatadine", "before sleep", "10"), s("hives", "before sleep")]),
    ("Loss of appetite and mild abdominal pain all day.", [s("loss of appetite", "all day"), s("abdominal pain", "all day", "mild")]),
    ("Took Maxpro 40 before breakfast on an empty stomach.", [m("Maxpro", "Esomeprazole", "before breakfast", "40, empty stomach")]),
    ("Mild chest congestion and productive cough this morning.", [s("chest congestion", "this morning", "mild"), s("productive cough", "this morning")]),
    ("Took one Tofen tablet at 9 PM.", [m("Tofen", "Ketotifen", "at 9 PM", "tablet")]),
    ("Swelling in both feet noticed after standing for long hours today.", [s("foot swelling", "today", "both feet")]),
    ("Had one Ace Plus around midnight for severe toothache.", [m("Ace Plus", "Paracetamol + Caffeine", "around midnight"), s("toothache", "around midnight", "severe")]),
    ("Feeling lightheaded since skipping lunch, no medicine taken.", [s("lightheadedness", "since skipping lunch")]),
    ("হ্যালো আচ্ছা সকালে নাপা খেয়েছিলাম আর মানে দুপুরে একটু পেট ব্যথা করছিল", [m("নাপা", "Paracetamol", "সকালে"), s("পেট ব্যথা", "দুপুরে", "একটু")]),
    ("সেক্লো ২০ খাইসি সকাল বেলা খালি পেটে", [m("সেক্লো", "Omeprazole", "সকাল বেলা", "খালি পেটে, ২০")]),
    ("মানে কালকে রাতে ওই যে অ্যালার্জি ট্যাবলেট নাম কি যেন এলাট্রল খেয়েছিলাম", [m("এলাট্রল", "Cetirizine", "কালকে রাতে")]),
    ("আরে ভাই মাথাটা প্রচণ্ড ঘুরতেছে সকাল থেকেই কিছু খাইনি এখনো", [s("মাথা ঘোরা", "সকাল থেকেই", "প্রচণ্ড")]),
    ("আব্বু সকাল ৮টায় কমেট ৫০০ খাইসে আর একটা কি যেন সাদা বড় বড়ি খাইসে", [m("কমেট", "Metformin", "সকাল ৮টায়", "৫০০"), m("সাদা বড় বড়ি", None, "সকাল ৮টায়")]),
    ("রাতে মনাস খাইসি ১০ এমজি কাশি এখনো আছে একটু একটু", [m("মনাস", "Montelukast", "রাতে", "১০ এমজি"), s("কাশি", "এখনো", "একটু একটু")]),
    ("অ্যামডোকাল ৫ সকালে নিসি প্রেসারের জন্য", [m("অ্যামডোকাল", "Amlodipine", "সকালে", "৫")]),
    ("দুপুরে খাবারের পর পেটটা ফাঁপা ফাঁপা লাগতেছে গ্যাস হইছে মনে হয়", [s("পেট ফাঁপা", "দুপুরে খাবারের পর"), s("গ্যাস", "দুপুরে খাবারের পর")]),
    ("সকালে ম্যাক্সপ্রো ৪০ আর রাতে সার্জেল ২০ খেলাম ভুলে দুইটাই গ্যাস্ট্রিকের", [m("ম্যাক্সপ্রো", "Esomeprazole", "সকালে", "৪০"), m("সার্জেল", "Esomeprazole", "রাতে", "২০")]),
    ("চোখে ড্রপ দিসি দুপুরে নামটা বোতলে লেখা ছোট বুঝতে পারতেছি না", [m("চোখের ড্রপ", None, "দুপুরে")]),
    ("হাত পা ঝিমঝিম করে বমি বমি লাগে কিছু খাই নাই আজকে", [s("হাত পা ঝিমঝিম", None), s("বমি বমি ভাব", None)]),
    ("সন্ধায় প্যানটোনিক্স ২০ আর একটা ভিটামিন ট্যাবলেট খাইসি", [m("প্যানটোনিক্স", "Pantoprazole", "সন্ধায়", "২০"), m("ভিটামিন ট্যাবলেট", None, "সন্ধায়")]),
    ("খুব জ্বর আসছিল দুপুরে তাই এইস ৫০০ খাইসি একটা", [s("জ্বর", "দুপুরে", "খুব"), m("এইস", "Paracetamol", "দুপুরে", "৫০০")]),
    ("ওই যে ঠান্ডার সিরাপটা খেয়েছি রাতে দুই চামচ নাম আদোভাস মনে হয়", [m("আদোভাস", None, "রাতে", "সিরাপ, দুই চামচ")]),
    ("রাত ৩টার সময় বুকে চাপ লাগছিল খুব ভয় পাইসি", [s("বুকে চাপ লাগা", "রাত ৩টার সময়")]),
    ("আজকে আর তেমন কোনো অসুবিধা নাই ওষুধও খাই নাই", []),
    ("সব নরমাল আজকে কোনো সমস্যা হয়নি", []),
    ("হ্যালো চেক চেক মাইক্রোফোন ঠিক আছে কিনা দেখছি", []),
    ("আজকের দিনটা খুব ভালো কেটেছে কোনো ব্যথা বা জ্বর নাই", []),
    ("কালকে সকালে প্রেসক্রিপশনটা খুঁজে বের করতে হবে", []),
    ("না আজকে কোনো রেকর্ড যোগ করার দরকার নাই", []),
    ("ফার্মেসি থেকে আম্মুর ওষুধগুলো কিনে আনতে হবে সন্ধ্যায়", []),
    ("No medicines or symptoms to record for today.", []),
    ("Just testing the voice input feature in Bangla and English.", []),
    ("ডাক্তার বলেছে আগামী সপ্তাহে আবার ব্লাড টেস্ট করাতে", []),
    ("আজকে রোদটা খুব বেশি ছিল বাইরে বের হয়েছিলাম দুপুরে", []),
    ("না এখন আর মাথাব্যথা বা কাশি কিছুই নাই সুস্থ আছি", []),
    ("I forgot to buy the medicine today so I didn't take anything.", []),
    ("রিপোর্টগুলো পিডিএফ করে রাখতে হবে ডাক্তারের জন্য", []),
    ("ঠিক আছে আজকের মতো রাখি ভালো থাকবেন", []),
]

EXTRA_FIX_EXAMPLES = [
    ("সকালে একটা নাপা খেয়েছি।", [m("নাপা", time="সকালে")]),
    ("নাস্তার আগে প্রোগাট ২০ খেয়েছি।", [m("প্রোগাট", time="নাস্তার আগে", note="২০")]),
    ("দুপুরে একটা Rabe 20 খেলাম।", [m("Rabe", time="দুপুরে", note="20")]),
    ("রাতে ঘুমানোর আগে একটা Fixal 120 নিয়েছি।", [m("Fixal", time="রাতে ঘুমানোর আগে", note="120")]),
    ("সকালে শুধু একটা প্যারাসিটামল খেয়েছি।", [m("প্যারাসিটামল", time="সকালে")]),
    ("বিকেলে একটা Tory 90 খেয়েছি।", [m("Tory", time="বিকেলে", note="90")]),
    ("আজকে একটা নীল রঙের ক্যাপসুল খেয়েছি।", [m("নীল রঙের ক্যাপসুল", time="আজকে")]),
    ("দুপুরে মক্সাসিল পাঁচশ আর রাতে Deflux খেয়েছি।", [m("মক্সাসিল", time="দুপুরে", note="পাঁচশ"), m("Deflux", time="রাতে")]),
    ("সকাল ৮টায় একটা Doxiva 200 নিয়েছি।", [m("Doxiva", time="সকাল ৮টায়", note="200")]),
    ("একটা Ciprox 500 খেয়েছি।", [m("Ciprox", note="500")]),

    ("Took one Napa in the morning.", [m("Napa", time="in the morning")]),
    ("Had Progut 20 before breakfast.", [m("Progut", time="before breakfast", note="20")]),
    ("Took Moxacil 500 after lunch.", [m("Moxacil", time="after lunch", note="500")]),
    ("Took Rabe 20 at night and Fixal 120 in the morning.", [m("Rabe", time="at night", note="20"), m("Fixal", time="in the morning", note="120")]),
    ("Took Napa in the morning, still have a mild fever.", [m("Napa", time="in the morning"), s("fever", note="mild")]),
    ("Had Seclo before lunch, still feeling slight heartburn.", [m("Seclo", time="before lunch"), s("heartburn", note="slight")]),
    ("Took one Tory 90 in the evening for knee pain.", [m("Tory", time="in the evening", note="90"), s("knee pain", time="in the evening")]),
    ("Took Deflux before dinner because of nausea and bloating.", [m("Deflux", time="before dinner"), s("nausea", time="before dinner"), s("bloating", time="before dinner")]),
    ("Took Doxiva 200 after breakfast, no wheezing today.", [m("Doxiva", time="after breakfast", note="200")]),
    ("Had Zis 500 at noon for a sore throat.", [m("Zis", time="at noon", note="500"), s("sore throat", time="at noon")]),
    ("Just took a blue capsule for headache, no time to rest.", [m("blue capsule"), s("headache")]),
    ("Had two spoonfuls of cough syrup before bed.", [m("cough syrup", time="before bed", note="two spoonfuls")]),
    ("Feeling dizzy since 2 PM, took an Orsaline packet.", [s("dizziness", time="since 2 PM"), m("Orsaline", note="one packet")]),
    ("Mild neck pain and fatigue since morning, haven't taken anything yet.", [s("neck pain", time="since morning", note="mild"), s("fatigue", time="since morning")]),

    ("সকালে নাস্তার আগে একটা Progut 20 খেয়েছি, এখনো বুক জ্বালাপোড়া করছে।", [m("Progut", time="সকালে নাস্তার আগে", note="20"), s("বুক জ্বালাপোড়া", time="এখনো")]),
    ("দুপুরে খাবার পর Doxiva 200 নিয়েছি আর রাতে মাথা ব্যথার জন্য একটা Tory খেয়েছি।", [m("Doxiva", time="দুপুরে খাবার পর", note="200"), m("Tory", time="রাতে"), s("মাথা ব্যথা", time="রাতে")]),
    ("মানে সকাল বেলা না পা এক্সট্রা একটা খাইসিলাম মাথাটা খুব ধরসিল এখন একটু কম", [m("না পা এক্সট্রা", time="সকাল বেলা"), s("মাথা ধরা", time="সকাল বেলা", note="খুব, এখন একটু কম")]),
    ("হ্যালো আচ্ছা রাতে ওই সেক লো বিশ খেয়েছি আর একটু বমি ভাব আছে এখনো", [m("সেক লো", time="রাতে", note="বিশ"), s("বমি ভাব", time="এখনো", note="একটু")]),
    ("সকাল থেকে একটু সর্দি আছে, রাতে Alatrol খাব ভাবছি যদি না কমে।", [s("সর্দি", time="সকাল থেকে", note="একটু")]),
    ("একটা পিনাডল খেয়েছি আর বুকে একটু জ্বালাপোড়া হচ্ছে।", [m("পিনাডল"), s("বুক জ্বালাপোড়া", note="একটু")]),
    ("Took Fixal 120 after lunch for runny nose and itchy eyes.", [m("Fixal", time="after lunch", note="120"), s("runny nose", time="after lunch"), s("itchy eyes", time="after lunch")]),
    ("Had Rabe 20 before dinner, still having mild gas.", [m("Rabe", time="before dinner", note="20"), s("gas", note="mild")]),
    ("Took two tablets of Ace 500 around 10 PM for body ache and chills.", [m("Ace", time="around 10 PM", note="500, two tablets"), s("body ache", time="around 10 PM"), s("chills", time="around 10 PM")]),
    ("আম্মুকে দুপুরে খাবার পর একটা Moxacil 500 দিয়েছি জ্বরের জন্য।", [m("Moxacil", time="দুপুরে খাবার পর", note="500"), s("জ্বর", time="দুপুরে খাবার পর")]),
    ("সকাল থেকে বাম কানে ব্যথা, দুপুরে একটা Ciprox 500 খেয়েছি।", [s("কানে ব্যথা", time="সকাল থেকে", note="বাম কানে"), m("Ciprox", time="দুপুরে", note="500")]),
    ("রাত ৯টায় Deflux খেয়েছি তবুও এখনো বমি বমি লাগছে।", [m("Deflux", time="রাত ৯টায়"), s("বমি বমি ভাব", time="এখনো")]),

    ("আজকে কোনো ওষুধ খাইনি।", []),
    ("আজকে আলহামদুলিল্লাহ ভালো আছি, কোনো সমস্যা নাই।", []),
    ("কোনো ওষুধ নিইনি আজ, শরীর একদম ঠিক আছে।", []),
    ("আজকে কোনো ব্যথা বা জ্বর নেই, ওষুধও খাইনি।", []),
    ("কিছু খাইনি আজকে, সব ঠিক আছে।", []),
    ("ওষুধ খাওয়া হয়নি আজ, এমনিতে সুস্থ আছি।", []),
    ("Didn't take any medicine today, everything is fine.", []),
    ("No symptoms and no medications to record today.", []),
    ("হ্যালো টেস্টিং করছি, আজকে কিছু রেকর্ড করার নেই।", []),
    ("সকালে ওষুধ কিনে এনেছি, এখনো খাইনি।", []),

    ("সকাল থেকে একটু মাথা ধরছে, রাতে নাপা খাব ভাবছি।", [s("মাথা ধরা", time="সকাল থেকে", note="একটু")]),
    ("পেটে হালকা গ্যাস লাগছে, না কমলে পরে সেকলো খাব।", [s("পেটে গ্যাস", note="হালকা")]),
    ("দুপুর থেকে হাঁচি হচ্ছে, সন্ধ্যায় ফেক্সো খেতে হবে ভাবছি।", [s("হাঁচি", time="দুপুর থেকে")]),
    ("হালকা জ্বর ভাব আছে, রাতে দরকার হলে এইস খাব।", [s("জ্বর ভাব", note="হালকা")]),
    ("বুক জ্বালাপোড়া করছে, ফার্মেসি থেকে সার্জেল আনতে হবে।", [s("বুক জ্বালাপোড়া")]),
    ("Mild headache since noon, thinking of taking Paracetamol tonight.", [s("headache", time="since noon", note="mild")]),

    ("সকালে খাবার পর Bizoran 5 খেয়েছি আর রাতে বুক জ্বলার জন্য একটা Progut খেয়েছি।", [m("Bizoran", time="সকালে খাবার পর", note="5"), m("Progut", time="রাতে"), s("বুক জ্বলা", time="রাতে")]),
    ("দুপুরে নাস্তার পর Moxacil 500 নিয়েছি আর সন্ধ্যায় জ্বরের জন্য একটা Napa খেয়েছি।", [m("Moxacil", time="দুপুরে নাস্তার পর", note="500"), m("Napa", time="সন্ধ্যায়"), s("জ্বর", time="সন্ধ্যায়")]),
    ("সকালে Fixal 120 খেয়েছি আর রাতে কাশির জন্য একটা Montair 10 খেয়েছি।", [m("Fixal", time="সকালে", note="120"), m("Montair", time="রাতে", note="10"), s("কাশি", time="রাতে")]),

    ("সকালে নাস্তার আগে একটা Progut 20 খেয়েছি, এখনো বুক জ্বালাপোড়া করছে আর একটু gas লাগছে।", [m("Progut", time="সকালে নাস্তার আগে", note="20"), s("বুক জ্বালাপোড়া", time="এখনো"), s("gas", time="এখনো", note="একটু")]),
    ("রাতে সেকলো ২০ খেয়েছি, এখনো হালকা মাথা ব্যথা আর একটু বমি ভাব আছে।", [m("সেকলো", time="রাতে", note="২০"), s("মাথা ব্যথা", time="এখনো", note="হালকা"), s("বমি ভাব", time="এখনো", note="একটু")]),
    ("Took Rabe 20 before breakfast, still having mild acidity and bloating.", [m("Rabe", time="before breakfast", note="20"), s("acidity", time="still", note="mild"), s("bloating", time="still")]),
]


_seen, ORIGINAL_EXAMPLES = set(), []
for t, recs in RAW_EXAMPLES + EXTRA_FIX_EXAMPLES:
    if t not in _seen:
        _seen.add(t); ORIGINAL_EXAMPLES.append((t, recs))
print(f"{len(ORIGINAL_EXAMPLES)} hand-written examples")

In [ ]:

BN_SYMPTOMS = [
    ("মাথা ব্যথা", ["করছে", "হচ্ছে"]), ("জ্বর", ["আছে", "এসেছে"]), ("কাশি", ["হচ্ছে", "আছে"]),
    ("শুকনো কাশি", ["হচ্ছে"]), ("সর্দি", ["লেগেছে", "আছে"]), ("গলা ব্যথা", ["করছে"]),
    ("বমি ভাব", ["হচ্ছে", "লাগছে"]), ("বমি", ["হয়েছে"]), ("পেট ব্যথা", ["করছে", "হচ্ছে"]),
    ("পেটে গ্যাস", ["হয়েছে"]), ("বুক জ্বালাপোড়া", ["করছে"]), ("মাথা ঘোরা", ["হচ্ছে"]),
    ("শরীর দুর্বল", ["লাগছে"]), ("পাতলা পায়খানা", ["হচ্ছে"]), ("কোমর ব্যথা", ["করছে"]),
    ("হাঁটু ব্যথা", ["করছে"]), ("পিঠে ব্যথা", ["হচ্ছে"]), ("বুকে ব্যথা", ["হচ্ছে"]),
    ("শ্বাসকষ্ট", ["হচ্ছে"]), ("নাক বন্ধ", ["হয়ে আছে"]), ("চুলকানি", ["হচ্ছে"]),
    ("গায়ে ব্যথা", ["করছে"]), ("দাঁত ব্যথা", ["করছে"]), ("কানে ব্যথা", ["হচ্ছে"]),
    ("বুক ধড়ফড়", ["করছে"]), ("কোষ্ঠকাঠিন্য", ["হচ্ছে"]), ("চোখ লাল", ["হয়ে আছে"]),
    ("হাত পা ঝিনঝিন", ["করছে"]), ("ঘাড় ব্যথা", ["করছে"]), ("পেট ফাঁপা", ["লাগছে"]),
    ("গা গরম", ["লাগছে"]), ("ঘুম কম", ["হচ্ছে"]), ("খিদে কম", ["লাগছে"]), ("ঠান্ডা", ["লেগেছে"]),
]
BANGLISH_SYMPTOMS = [
    ("headache", ["হচ্ছে"]), ("gas", ["হচ্ছে"]), ("acidity", ["হচ্ছে"]), ("fever", ["আছে"]),
    ("cough", ["হচ্ছে"]), ("weakness", ["লাগছে"]), ("back pain", ["হচ্ছে"]), ("vomiting", ["হচ্ছে"]),
    ("loose motion", ["হচ্ছে"]), ("allergy", ["হচ্ছে"]), ("chest pain", ["হচ্ছে"]),
    ("bloating", ["হচ্ছে"]), ("dizziness", ["হচ্ছে"]), ("sore throat", ["হচ্ছে"]), ("pain", ["হচ্ছে"]),
]
EN_SYMPTOMS = [
    "headache", "fever", "cough", "dry cough", "sore throat", "nausea", "stomach pain", "back pain",
    "dizziness", "runny nose", "chest pain", "fatigue", "body ache", "heartburn", "diarrhea",
    "joint pain", "shortness of breath", "itching", "toothache", "knee pain", "neck pain",
    "bloating", "acidity", "blocked nose", "weakness", "vomiting", "ear pain", "insomnia",
]
BN_INTENSITY = ["একটু", "হালকা", "খুব", "প্রচণ্ড", "অনেক", "মাঝে মাঝে", "বেশ"]
EN_INTENSITY = ["mild", "severe", "slight", "bad", "a little", "constant"]
BN_SYM_TIME = ["সকাল থেকে", "দুপুর থেকে", "বিকেল থেকে", "রাত থেকে", "গতকাল থেকে", "দুই দিন ধরে",
               "এখনো", "কাল রাত থেকে", "ঘুম থেকে উঠে", "খাওয়ার পর থেকে", "আজ সকালে", "তিন দিন ধরে"]
EN_SYM_TIME = ["since morning", "since last night", "since yesterday", "for two days", "today",
               "still", "since lunch", "this evening", "after waking up"]
BN_MED_TIME = ["সকালে", "সকালে নাস্তার পর", "নাস্তার আগে", "দুপুরে খাবার পর", "রাতে ঘুমানোর আগে",
               "সন্ধ্যায়", "দুপুরে", "রাতে", "সকাল ৯টায়", "আজ সকালে"]
EN_MED_TIME = ["in the morning", "after breakfast", "before dinner", "at night", "after lunch",
               "this morning", "at 9 am", "before bed"]
STRENGTH = ["20", "500", "10", "40", "120", "5", "৫০০", "২০", "১০", "650", "2.5"]
BRANDS_LATIN = ["Napa", "Ace", "Seclo", "Sergel", "Maxpro", "Esonix", "Monas", "Fexo", "Alatrol",
                "Bizoran", "Comet", "Pantonix", "Losectil", "Zimax", "Tufnil", "Progut", "Rabe",
                "Fixal", "Doxiva", "Tory", "Deflux", "Ciprox", "Napa Extra", "Montair", "Rolac"]
BRANDS_BN = ["নাপা", "সেকলো", "মোনাস", "ফেক্সো", "এইস", "সারজেল", "প্যারাসিটামল", "ম্যাক্সপ্রো",
             "অ্যালাট্রল", "নাপা এক্সট্রা", "প্রোগাট", "এসোনিক্স"]
BN_MED_VERB = ["খেয়েছি", "খেলাম", "নিয়েছি", "খাইছি", "খেয়েছিলাম"]
BN_FILLERS = ["আচ্ছা", "মানে", "হ্যালো", "ইয়ে", "শোনো"]
EN_FILLERS = ["um", "so", "okay", "well"]

def invented_brand(rng):
    syll = ["ra", "zo", "fen", "lo", "ke", "mi", "tor", "vex", "ci", "dal", "ne", "pra", "sul", "xa", "mo"]
    return "".join(rng.choice(syll) for _ in range(rng.choice([2, 2, 3]))).capitalize()

def join_bn(parts, rng):
    if len(parts) == 1: return parts[0]
    conj = rng.choice(["আর", "আর", "সাথে", "এবং"])
    return ", ".join(parts[:-1]) + f" {conj} " + parts[-1]

def join_en(parts):
    return parts[0] if len(parts) == 1 else ", ".join(parts[:-1]) + " and " + parts[-1]

def pick_counts(rng):
    n_sym = rng.choices([0, 1, 2, 3, 4], weights=[10, 22, 30, 26, 12])[0]
    n_med = rng.choices([0, 1, 2], weights=[35, 50, 15])[0]
    if n_sym == 0 and n_med == 0:
        n_med = 1
    return n_sym, n_med

def gen_bn(rng, banglish=False):
    n_sym, n_med = pick_counts(rng)
    pool = BANGLISH_SYMPTOMS + BN_SYMPTOMS[:12] if banglish else BN_SYMPTOMS
    syms = rng.sample(pool, n_sym)
    clauses = []

    if n_sym:
        t = rng.choice([None] + BN_SYM_TIME * 2)
        parts, recs = [], []
        for label, verbs in syms:
            inten = rng.choice([None, None] + BN_INTENSITY)
            parts.append(" ".join(x for x in [inten, label, rng.choice(verbs)] if x))
            recs.append(s(label, t, inten))
        clauses.append((" ".join(x for x in [t, join_bn(parts, rng)] if x), recs))

    for _ in range(n_med):
        brand = rng.choice(BRANDS_BN if rng.random() < 0.35 else BRANDS_LATIN + [invented_brand(rng)])
        t = rng.choice([None] + BN_MED_TIME * 2)
        st = rng.choice([None, None] + STRENGTH)
        count = rng.choice(["একটা", "", ""])
        txt = " ".join(x for x in [t, count, brand, st, rng.choice(BN_MED_VERB)] if x)
        clauses.append((txt, [m(brand, time=t, note=st)]))

    if rng.random() < 0.25:
        neg = rng.choice(BN_SYMPTOMS)[0]
        if all(neg != r["name"] for _, rs in clauses for r in rs):
            clauses.append((rng.choice([f"তবে {neg} নেই", f"আজ {neg} হয়নি", f"এখন আর {neg} নেই"]), []))
    if rng.random() < 0.15:
        clauses.append((rng.choice(["দুপুরে ভাত খেয়েছি", "সকালে রুটি খেয়েছি", "অনেক পানি খেয়েছি"]), []))
    if rng.random() < 0.12:
        clauses.append((f"রাতে {rng.choice(BRANDS_LATIN)} খাব ভাবছি", []))

    rng.shuffle(clauses)
    sep = rng.choice(["। ", ", ", " "])
    text = sep.join(c for c, _ in clauses).strip() + rng.choice(["।", "।", ""])
    if rng.random() < 0.2:
        text = rng.choice(BN_FILLERS) + " " + text.replace("।", "")
    records = [r for _, rs in clauses for r in rs]
    return text, records

def gen_en(rng):
    n_sym, n_med = pick_counts(rng)
    syms = rng.sample(EN_SYMPTOMS, n_sym)
    clauses = []
    if n_sym:
        t = rng.choice([None] + EN_SYM_TIME * 2)
        parts, recs = [], []
        for label in syms:
            inten = rng.choice([None, None] + EN_INTENSITY)
            parts.append(" ".join(x for x in [inten, label] if x))
            recs.append(s(label, t, inten))
        lead = rng.choice(["I have", "still having", "got", "I've had", "having"])
        clauses.append((" ".join(x for x in [lead, join_en(parts), t] if x), recs))
    for _ in range(n_med):
        brand = rng.choice(BRANDS_LATIN + [invented_brand(rng)])
        t = rng.choice([None] + EN_MED_TIME * 2)
        st = rng.choice([None, None] + [x for x in STRENGTH if x.isascii()])
        txt = " ".join(x for x in [rng.choice(["took", "had", "took one"]), brand, st, t] if x)
        clauses.append((txt, [m(brand, time=t, note=st)]))
    if rng.random() < 0.25:
        neg = rng.choice(EN_SYMPTOMS)
        if all(neg != r["name"] for _, rs in clauses for r in rs):
            clauses.append((f"no {neg} today", []))
    if rng.random() < 0.1:
        clauses.append((f"might take {rng.choice(BRANDS_LATIN)} tonight", []))
    rng.shuffle(clauses)
    text = ", ".join(c for c, _ in clauses)
    text = text[0].upper() + text[1:] + "."
    if rng.random() < 0.15:
        text = rng.choice(EN_FILLERS) + " " + text[0].lower() + text[1:-1]
    return text, [r for _, rs in clauses for r in rs]

def generate(n, seed):
    rng = random.Random(seed)
    out, seen = [], set()
    while len(out) < n:
        kind = rng.choices(["bn", "banglish", "en"], weights=[50, 25, 25])[0]
        ex = gen_en(rng) if kind == "en" else gen_bn(rng, banglish=(kind == "banglish"))
        if ex[0] not in seen:
            seen.add(ex[0]); out.append(ex)
    return out

SYNTH_TRAIN = generate(SYNTH_TRAIN_N, SEED)
SYNTH_VAL = [e for e in generate(SYNTH_VAL_N * 3, SEED + 1) if e[0] not in {t for t, _ in SYNTH_TRAIN}][:SYNTH_VAL_N]

from collections import Counter
dist = Counter(sum(r["type"] == "symptom" for r in recs) for _, recs in SYNTH_TRAIN)
print("symptoms per synthetic example:", dict(sorted(dist.items())))
print("\nSamples (eyeball these — they are your training data):")
for t, recs in SYNTH_TRAIN[:8]:
    print(" •", t); print("   →", json.dumps({"records": recs}, ensure_ascii=False))

In [ ]:
rng = random.Random(SEED)
orig = list(ORIGINAL_EXAMPLES); rng.shuffle(orig)
n_val = max(40, int(len(orig) * ORIG_VAL_FRAC))
ORIG_VAL, ORIG_TRAIN = orig[:n_val], orig[n_val:]

TRAIN_EXAMPLES = ORIG_TRAIN * ORIG_UPWEIGHT + SYNTH_TRAIN
rng.shuffle(TRAIN_EXAMPLES)
assert not ({t for t, _ in ORIG_VAL} & {t for t, _ in TRAIN_EXAMPLES})

REQUIRED_KEYS = {"type", "name", "generic_suggestion", "time_phrase", "note"}
def write_jsonl(examples, path):
    with open(path, "w", encoding="utf-8") as f:
        for text, records in examples:
            for r in records:
                assert set(r) == REQUIRED_KEYS and r["type"] in ("medicine", "symptom")
                assert r["generic_suggestion"] is None and r["name"].strip()
            f.write(json.dumps({"conversations": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": text},
                {"role": "assistant", "content": json.dumps({"records": records}, ensure_ascii=False)},
            ]}, ensure_ascii=False) + "\n")

TRAIN_PATH, VAL_PATH = "medidiary_train_v2.jsonl", "medidiary_val_v2.jsonl"
write_jsonl(TRAIN_EXAMPLES, TRAIN_PATH)
write_jsonl(ORIG_VAL, VAL_PATH)
print(f"train={len(TRAIN_EXAMPLES)} (orig {len(ORIG_TRAIN)}×{ORIG_UPWEIGHT} + synth {len(SYNTH_TRAIN)}) | "
      f"val: orig={len(ORIG_VAL)} synth={len(SYNTH_VAL)}")

In [ ]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name = BASE_MODEL,
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit = True,
    load_in_8bit = False,
    full_finetuning = False,
)
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers = False,
    finetune_language_layers = True,
    finetune_attention_modules = True,
    finetune_mlp_modules = True,
    r = LORA_R,
    lora_alpha = LORA_ALPHA,
    lora_dropout = 0,
    bias = "none",
    random_state = SEED,
)

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template, standardize_data_formats, train_on_responses_only
from trl import SFTTrainer, SFTConfig

tokenizer = get_chat_template(tokenizer, chat_template = "gemma-3")

def load(path):
    ds = standardize_data_formats(load_dataset("json", data_files = path, split = "train"))
    return ds.map(lambda ex: {"text": [
        tokenizer.apply_chat_template(c, tokenize = False, add_generation_prompt = False).removeprefix("<bos>")
        for c in ex["conversations"]]}, batched = True)

train_dataset, val_dataset = load(TRAIN_PATH), load(VAL_PATH)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        dataset_num_proc = 2,
        packing = False,
        max_length = MAX_SEQ_LEN,
        per_device_train_batch_size = BATCH,
        gradient_accumulation_steps = GRAD_ACC,
        warmup_ratio = 0.05,
        num_train_epochs = EPOCHS,
        learning_rate = LR,
        lr_scheduler_type = "cosine",
        weight_decay = 0.01,
        optim = "adamw_8bit",
        logging_steps = 10,
        eval_strategy = "epoch",
        save_strategy = "epoch",
        save_total_limit = 2,
        load_best_model_at_end = True,
        metric_for_best_model = "eval_loss",
        seed = SEED,
        output_dir = "outputs_v2",
        report_to = "none",
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n",
)

row0 = tokenizer.decode([x for x in trainer.train_dataset[0]["labels"] if x != -100])
print("=== Trained-on tokens of row 0 (should be ONE JSON reply) ===\n", row0)
assert row0.count('{"records":') == 1, "Expected exactly one response per row (packing must be off)."
print("optimizer steps:", len(trainer.get_train_dataloader()) // GRAD_ACC * EPOCHS)

In [ ]:
trainer_stats = trainer.train()
print(f"Training finished in {trainer_stats.metrics['train_runtime']/60:.1f} min")

In [ ]:
import re, json, unicodedata

def extract_json(raw):
    """First balanced {...} block — mirrors the app's ModelResponseParser."""
    raw = raw.replace("```json", "").replace("```", "")
    start = raw.find("{")
    if start < 0: return None
    depth, in_str, esc = 0, False, False
    for i, c in enumerate(raw[start:], start):
        if in_str:
            if esc: esc = False
            elif c == "\\": esc = True
            elif c == '"': in_str = False
            continue
        if c == '"': in_str = True
        elif c == "{": depth += 1
        elif c == "}":
            depth -= 1
            if depth == 0:
                try: return json.loads(raw[start:i + 1])
                except Exception: return None
    return None

BN_DIGITS = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")
def norm(x):
    if x is None: return ""
    x = unicodedata.normalize("NFC", str(x)).translate(BN_DIGITS).lower()
    x = x.replace("‌", "").replace("‍", "")
    return re.sub(r"[\s\-_,।.!?]+", "", x)

def lenient_eq(a, b):
    a, b = norm(a), norm(b)
    if not a or not b: return False
    return a == b or (min(len(a), len(b)) >= 2 and (a in b or b in a))

def match(gold, pred, lenient):
    """Greedy one-to-one matching of names; returns list of (gold_rec, pred_rec)."""
    used, pairs = set(), []
    for g in gold:
        for j, p in enumerate(pred):
            if j in used: continue
            if (lenient_eq(g["name"], p["name"]) if lenient else norm(g["name"]) == norm(p["name"])):
                used.add(j); pairs.append((g, p)); break
    return pairs

def score(examples, predict, label, show_failures=8):
    stats = {k: {"gold": 0, "pred": 0, "strict": 0, "lenient": 0, "time_ok": 0} for k in ("medicine", "symptom")}
    valid, failures, by_n = 0, [], {}
    for text, gold in examples:
        raw = predict(text)
        parsed = extract_json(raw)
        recs = parsed.get("records") if isinstance(parsed, dict) else None
        if isinstance(recs, list): valid += 1
        else: recs = []
        recs = [r for r in recs if isinstance(r, dict) and isinstance(r.get("name"), str)]
        ok = True
        for k in stats:
            g = [r for r in gold if r["type"] == k]; p = [r for r in recs if r.get("type") == k]
            strict, lenp = match(g, p, False), match(g, p, True)
            st = stats[k]
            st["gold"] += len(g); st["pred"] += len(p); st["strict"] += len(strict); st["lenient"] += len(lenp)
            st["time_ok"] += sum(norm(a.get("time_phrase")) == norm(b.get("time_phrase")) for a, b in lenp)
            if len(lenp) != len(g) or len(p) != len(g): ok = False
            if k == "symptom":
                n = len(g); d = by_n.setdefault(n, [0, 0]); d[0] += len(lenp); d[1] += n
        if not ok: failures.append((text, gold, raw))

    report = {"set": label, "n": len(examples), "valid_json": valid / max(1, len(examples))}
    print(f"\n=== {label}  (n={len(examples)}) ===   valid JSON {valid}/{len(examples)}")
    for k, st in stats.items():
        for mode in ("strict", "lenient"):
            P = st[mode] / max(1, st["pred"]); R = st[mode] / max(1, st["gold"]); F = 2 * P * R / max(1e-9, P + R)
            report[f"{k}_{mode}_P"], report[f"{k}_{mode}_R"], report[f"{k}_{mode}_F1"] = P, R, F
            print(f"  {k:8s} {mode:7s}  P {P:6.1%}  R {R:6.1%}  F1 {F:6.1%}   ({st[mode]}/{st['gold']} gold, {st['pred']} predicted)")
        print(f"  {k:8s} time_phrase correct on matched: {st['time_ok']}/{max(1, st['lenient'])}")
    print("  symptom recall by #symptoms in transcript:",
          {n: f"{a}/{b}" for n, (a, b) in sorted(by_n.items()) if b})
    for text, gold, raw in failures[:show_failures]:
        print(f"\n  ✗ {text}\n    GOLD: {json.dumps(gold, ensure_ascii=False)}\n    PRED: {raw.strip()[:400]}")
    report["failures"] = [{"text": t, "gold": g, "pred": r} for t, g, r in failures]
    return report

In [ ]:
FastModel.for_inference(model)

def predict_hf(transcript):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": transcript}]
    prompt = tokenizer.apply_chat_template(messages, tokenize = False, add_generation_prompt = True).removeprefix("<bos>")
    inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens = 384, do_sample = False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens = True).strip()

STRESS = [
    "সকালে নাস্তার আগে একটা Esonix 20 খেয়েছি, এখনো বুক জ্বালাপোড়া করছে আর একটু gas লাগছে।",
    "গতকাল রাত থেকে জ্বর, মাথা ব্যথা, গলা ব্যথা আর হালকা কাশি হচ্ছে, সকালে একটা Napa খেয়েছি।",
    "Took Napa in the morning, still have a mild fever, body ache and a runny nose.",
    "সকাল থেকে একটু সর্দি আছে, রাতে Alatrol খাব ভাবছি যদি না কমে।",
    "আজকে কোনো ওষুধ খাইনি, সব ঠিক আছে।",
    "মানে দুপুর থেকে পেট ব্যথা পাতলা পায়খানা আর বমি বমি লাগছে খুব দুর্বল লাগছে",
    "জ্বর নেই, তবে খুব কাশি হচ্ছে আর বুকে ব্যথা।",
]
print("=== Stress tests ===")
for t in STRESS:
    print("IN :", t); print("OUT:", predict_hf(t), "\n")

REPORTS = [score(ORIG_VAL, predict_hf, "HF fp16 — held-out hand-written (v1 had 48.1% strict symptom recall on its own 42-example split)"),
           score(SYNTH_VAL, predict_hf, "HF fp16 — held-out synthetic (multi-symptom)")]

In [ ]:
import glob, shutil

model.save_pretrained(f"{RUN_NAME}_lora")
tokenizer.save_pretrained(f"{RUN_NAME}_lora")
model.save_pretrained_gguf(f"{RUN_NAME}_gguf", tokenizer, quantization_method = QUANT)

ggufs = [p for p in glob.glob(f"{RUN_NAME}_gguf*/**/*.gguf", recursive = True) if QUANT.upper() in p.upper()]
assert ggufs, f"No {QUANT} GGUF found: {glob.glob('**/*.gguf', recursive=True)}"
GGUF_PATH = ggufs[0]
APP_NAME = "medidiary-gemma.gguf"
shutil.copy(GGUF_PATH, APP_NAME)
print("GGUF:", GGUF_PATH, f"({os.path.getsize(GGUF_PATH)/1e9:.2f} GB) → copied as", APP_NAME)

if SAVE_TO_DRIVE:
    shutil.copy(APP_NAME, f"{DRIVE_DIR}/{APP_NAME}")
    shutil.copytree(f"{RUN_NAME}_lora", f"{DRIVE_DIR}/{RUN_NAME}_lora", dirs_exist_ok = True)
    for p in (TRAIN_PATH, VAL_PATH): shutil.copy(p, DRIVE_DIR)
    print("Saved to", DRIVE_DIR, os.listdir(DRIVE_DIR))

In [ ]:
import subprocess, shutil, glob, tempfile

cands = [p for p in glob.glob("/root/.unsloth/llama.cpp/**/llama-completion", recursive = True)
         + glob.glob("/content/**/llama-completion", recursive = True) if os.access(p, os.X_OK)]
if not cands and shutil.which("llama-completion"): cands = [shutil.which("llama-completion")]

def app_prompt(transcript):
    return f"<start_of_turn>user\n{SYSTEM_PROMPT.strip()}\n\n{transcript.strip()}<end_of_turn>\n<start_of_turn>model\n"

def predict_gguf(transcript):
    with tempfile.NamedTemporaryFile("w", suffix = ".txt", delete = False, encoding = "utf-8") as f:
        f.write(app_prompt(transcript))
    r = subprocess.run([cands[0], "-m", APP_NAME, "-f", f.name, "-n", "384", "--temp", "0", "-c", str(MAX_SEQ_LEN),
                        "-no-cnv", "--no-display-prompt", "-t", str(os.cpu_count())],
                       capture_output = True, text = True, timeout = 300, stdin = subprocess.DEVNULL)
    if r.returncode != 0:
        raise RuntimeError(f"llama-completion failed ({r.returncode}):\n{r.stderr[-2000:]}")
    out = r.stdout
    i = out.find('{"records"')
    return out[i:] if i >= 0 else out

if not cands:
    print("llama-completion binary not found — skipping GGUF check. The fp16 metrics above still apply.")
else:
    print("Using", cands[0])
    print("Sample:", predict_gguf(STRESS[1])[:400])
    REPORTS.append(score(ORIG_VAL[:GGUF_EVAL_N], predict_gguf, f"GGUF {QUANT} — held-out hand-written (first {GGUF_EVAL_N})"))
    print("\nIf GGUF recall is far below fp16, re-run with QUANT = 'q8_0'.")

In [ ]:
summary = [{k: v for k, v in r.items() if k != "failures"} for r in REPORTS]
with open("eval_report_v2.json", "w", encoding = "utf-8") as f:
    json.dump(REPORTS, f, ensure_ascii = False, indent = 1)
if SAVE_TO_DRIVE: shutil.copy("eval_report_v2.json", DRIVE_DIR)
for r in summary:
    print(f"{r['set'][:60]:60s}  symptom R strict {r['symptom_strict_R']:.1%} / lenient {r['symptom_lenient_R']:.1%}"
          f"   medicine R {r['medicine_lenient_R']:.1%}")

### Put it on the phone

```bash
adb push medidiary-gemma.gguf /sdcard/Android/data/com.medidiary.app/files/models/medidiary-gemma.gguf
```

then **Settings → Load model → Re-scan** (or download it from Drive on the phone and use **Load model → Import…**).
App settings that matter for this model: *Use GGUF chat template* **on**, context **1024**.

### If symptom recall is still not good enough
* Read the `✗` failures above. If many are wording differences (lenient ✓, strict ✗), the model is fine — the review screen lets users edit names.
* Add real transcripts from your team recordings to `EXTRA_FIX_EXAMPLES`, following the labeling rule (name = words as spoken).
* Raise `SYNTH_TRAIN_N` to 2000 or `EPOCHS` to 4; watch eval loss — if it rises after epoch 2, keep 2–3.